# Scenario-Conditioned Degradation Forecasting and Knowledge-Aware Risk-Budgeted Continual Adaptation for Network Intrusion Detection

**Runnable reference implementation** of the supplied methodology: chronological flow preparation, a base IDS, drift-state monitoring, SCDF-style scenario-conditioned degradation forecasting, SKTG-style concept tension graph, RBPA-style risk-budgeted action selection, delayed labels, safety gate, and rollback audit.

> **Important research note:** This notebook implements a practical, reproducible reference pipeline. The manuscript's complete rolling-origin shadow-update calibration, block-bootstrap guarantees, exact dataset-day partitions, and 10-seed statistical protocol require substantial dataset-specific configuration and compute; they are not silently claimed as fully reproduced here. The notebook reports measured results from the data you provide and labels synthetic controlled-drift experiments separately.

## 0. Runtime and configuration
Run in Google Colab. For CIC-IDS2017 / CSE-CIC-IDS2018, upload corrected CSV flow files or place them in Google Drive. The loader accepts one CSV or a directory of CSVs. Keep raw files out of the notebook output.

In [ ]:
# If needed in Colab:
# from google.colab import drive
# drive.mount('/content/drive')

import os, glob, time, math, copy, random, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve)
from sklearn.linear_model import QuantileRegressor, Ridge
from sklearn.isotonic import IsotonicRegression
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances
from sklearn.utils.class_weight import compute_class_weight
import scipy
from scipy.special import expit
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# EDIT THIS PATH. Can be a CSV file or a folder containing CSV files.
DATA_PATH = '/content/drive/MyDrive/CIC-IDS2017'
# If using a single uploaded file, e.g. /content/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
MAX_ROWS = None              # set e.g. 300_000 for a quick smoke test; None uses all rows
WINDOW_MINUTES = 5
MIN_WINDOW_SUPPORT = 50
LABEL_DELAY_WINDOWS = 3
HISTORY_LENGTH = 12
EPOCHS = 12
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
THRESHOLD = 0.5
RESOURCE_BUDGET = 0.65
FORGETTING_BUDGET = 0.15
HARM_TOLERANCE = 0.05
MIN_CLASS_SUPPORT_FOR_UPDATE = 30

## 1. Load and inspect flow records
The code detects common CIC-IDS / CSE-CIC-IDS / UNSW-NB15 label and timestamp columns. If automatic detection is wrong, set `LABEL_COL` and `TIME_COL` manually after viewing the printed columns. Flow IDs, IP addresses, timestamps and labels are excluded from model inputs.

In [ ]:
def read_flow_files(path, max_rows=None):
    p = Path(path)
    if p.is_file() and p.suffix.lower() == '.csv':
        files = [p]
    elif p.is_dir():
        files = sorted(p.rglob('*.csv'))
    else:
        files = sorted(Path('/content').glob('*.csv')) if str(path).strip() == '' else []
    if not files:
        raise FileNotFoundError(f'No CSV files found at {path}. Set DATA_PATH to a CSV or folder.')
    frames=[]
    for f in files:
        try:
            df = pd.read_csv(f, low_memory=False)
            if len(df):
                df['_source_file'] = f.name
                frames.append(df)
                print(f'Loaded {f.name}: {df.shape}')
        except Exception as e:
            print('Skipping', f.name, '->', str(e)[:160])
    if not frames: raise ValueError('No readable non-empty CSV files.')
    data = pd.concat(frames, ignore_index=True, sort=False)
    data.columns = [str(c).strip().replace('\ufeff','') for c in data.columns]
    if max_rows and len(data) > max_rows:
        # Keep an early chronological slice after sorting when a timestamp is detectable.
        data = data.iloc[:max_rows].copy()
    return data

df_raw = read_flow_files(DATA_PATH, MAX_ROWS)
print('Combined shape:', df_raw.shape)
print('Columns:', list(df_raw.columns))
display(df_raw.head())

## 2. Label/time mapping and basic cleaning
Binary target convention: benign = 0, attack = 1. Family labels are retained for concept-level analysis. Timestamps are used only for ordering/window construction, never as model features. Dataset-specific label spellings are normalized.

In [ ]:
def find_col(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lookup: return lookup[cand.lower()]
    for c in columns:
        cl = str(c).strip().lower()
        if any(k in cl for k in candidates): return c
    return None

LABEL_COL = find_col(df_raw.columns, [
    'Label','label','class','attack_cat','attack category','category','target'
])
TIME_COL = find_col(df_raw.columns, [
    'Timestamp','timestamp','Flow Start Time','flow start time','stime','StartTime','start_time'
])
DURATION_COL = find_col(df_raw.columns, ['Flow Duration','flow_duration','dur','duration'])
print('Detected label:', LABEL_COL, '| timestamp:', TIME_COL, '| duration:', DURATION_COL)
if LABEL_COL is None:
    raise ValueError('Could not detect a label column. Set LABEL_COL manually to the exact column name.')
if TIME_COL is None:
    print('WARNING: No timestamp found. Chronological order will use source-file order and row order; this is weaker than timestamp-based evaluation.')

def normalize_label(v):
    s = str(v).strip().lower()
    benign_tokens = {'benign','normal','0','normal traffic','background'}
    return 0 if s in benign_tokens else 1

df = df_raw.copy()
df = df.loc[:, ~pd.Index(df.columns).duplicated()].copy()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna(subset=[LABEL_COL]).copy()
df['_family'] = df[LABEL_COL].astype(str).str.strip()
df['_y'] = df[LABEL_COL].map(normalize_label).astype(int)

if TIME_COL is not None:
    # CIC timestamps often use day/month/year and seconds; pandas handles most common formats.
    parsed = pd.to_datetime(df[TIME_COL], errors='coerce', dayfirst=True, utc=True)
    if parsed.notna().mean() >= 0.5:
        df['_time'] = parsed
    else:
        print('Timestamp parsing unreliable; preserving file/row order.')
        df['_time'] = np.arange(len(df))
else:
    df['_time'] = np.arange(len(df))

# Remove exact duplicate raw records only; don't deduplicate based solely on features/label.
df = df.drop_duplicates().copy()
df = df.sort_values('_time', kind='stable').reset_index(drop=True)
print('Cleaned shape:', df.shape)
print('Binary labels:', df['_y'].value_counts().to_dict())
print('Top families:')
display(df['_family'].value_counts().head(15).to_frame('count'))

## 3. Before-preprocessing visualisation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df['_y'].value_counts().sort_index().rename(index={0:'Benign',1:'Attack'}).plot(kind='bar', ax=ax[0])
ax[0].set_title('Binary label distribution before preprocessing'); ax[0].set_ylabel('Flows')
df['_family'].value_counts().head(12).plot(kind='bar', ax=ax[1])
ax[1].set_title('Top family labels before preprocessing'); ax[1].tick_params(axis='x', rotation=55)
plt.tight_layout(); plt.show()

## 4. Chronological train/validation/test partitions
The default split is 70/15/15 in chronological order. This is a generic safe default, not the manuscript's exact day/session split. For strict replication, set partition boundaries to the specified capture sessions before running this cell. Preprocessing is fitted on training rows only.

In [ ]:
n = len(df)
i_train = int(n * 0.70)
i_val = int(n * 0.85)
train_df = df.iloc[:i_train].copy()
val_df = df.iloc[i_train:i_val].copy()
test_df = df.iloc[i_val:].copy()
if min(len(train_df), len(val_df), len(test_df)) == 0:
    raise ValueError('A chronological partition is empty; dataset is too small.')
print('Train/val/test rows:', len(train_df), len(val_df), len(test_df))
print('Train time range:', train_df['_time'].min(), '->', train_df['_time'].max())
print('Val time range:', val_df['_time'].min(), '->', val_df['_time'].max())
print('Test time range:', test_df['_time'].min(), '->', test_df['_time'].max())

# Exclude labels, raw time, IDs, endpoints, source filenames, and obvious leakage columns.
exclude_exact = {LABEL_COL, '_family', '_y', '_time', '_source_file'}
exclude_tokens = ['flow id','src ip','source ip','dst ip','destination ip','source address',
                  'destination address','srcip','dstip','filename','file name','timestamp',
                  'attack_cat','attack category']
candidate_cols = []
for c in df.columns:
    cl = str(c).strip().lower()
    if c in exclude_exact or cl in {'label','class','target'}: continue
    if any(t in cl for t in exclude_tokens): continue
    if pd.api.types.is_datetime64_any_dtype(df[c]): continue
    candidate_cols.append(c)

# Drop columns that are entirely missing in training.
candidate_cols = [c for c in candidate_cols if train_df[c].notna().any()]
if not candidate_cols: raise ValueError('No usable features detected.')
X_train_raw = train_df[candidate_cols].copy()
X_val_raw = val_df[candidate_cols].copy()
X_test_raw = test_df[candidate_cols].copy()

# Numeric features plus low-cardinality categoricals (e.g., protocol); ignore high-cardinality text.
num_cols = [c for c in candidate_cols if pd.api.types.is_numeric_dtype(train_df[c])]
cat_cols = [c for c in candidate_cols if c not in num_cols and train_df[c].nunique(dropna=True) <= 30]
ignored_cols = [c for c in candidate_cols if c not in num_cols and c not in cat_cols]
print('Numeric features:', len(num_cols), '| categorical:', len(cat_cols), '| ignored high-cardinality:', len(ignored_cols))
if not num_cols and not cat_cols: raise ValueError('No numeric or low-cardinality categorical input features remain.')

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)
transformers=[]
if num_cols:
    transformers.append(('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                                         ('scaler', RobustScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                         ('onehot', ohe)]), cat_cols))
preprocessor = ColumnTransformer(transformers, remainder='drop', verbose_feature_names_out=False)
X_train = preprocessor.fit_transform(X_train_raw).astype(np.float32)
X_val = preprocessor.transform(X_val_raw).astype(np.float32)
X_test = preprocessor.transform(X_test_raw).astype(np.float32)
y_train = train_df['_y'].to_numpy(np.int64)
y_val = val_df['_y'].to_numpy(np.int64)
y_test = test_df['_y'].to_numpy(np.int64)
family_train = train_df['_family'].to_numpy(str)
family_val = val_df['_family'].to_numpy(str)
family_test = test_df['_family'].to_numpy(str)
print('Prepared matrices:', X_train.shape, X_val.shape, X_test.shape)

## 5. After-preprocessing visualisation and window construction
Each flow is assigned to a five-minute window using flow completion time when a duration column is available. Windows with fewer than `MIN_WINDOW_SUPPORT` flows remain in the stream but have unreliable summary statistics; they are not labelled benign. This practical implementation also supports row-chunk windows when timestamps cannot be parsed.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
pd.Series(y_train).value_counts().sort_index().rename(index={0:'Benign',1:'Attack'}).plot(kind='bar', ax=ax[0])
ax[0].set_title('Training labels after split'); ax[0].set_ylabel('Flows')
missing_rate = X_train_raw.isna().mean().sort_values(ascending=False).head(15)
missing_rate.plot(kind='bar', ax=ax[1]); ax[1].set_title('Top training feature missing rates before imputation')
ax[1].tick_params(axis='x', rotation=55)
plt.tight_layout(); plt.show()

def make_windows(frame, X, y, families, window_minutes=5, min_support=50):
    tmp = frame[['_time']].copy().reset_index(drop=True)
    tmp['_row'] = np.arange(len(tmp))
    tmp['_y'] = y
    tmp['_family'] = families
    if isinstance(tmp['_time'].dtype, pd.DatetimeTZDtype) or pd.api.types.is_datetime64_any_dtype(tmp['_time']):
        dur = pd.to_numeric(frame[DURATION_COL], errors='coerce').fillna(0).to_numpy() if DURATION_COL else np.zeros(len(frame))
        # CIC duration commonly microseconds; use it only if the magnitude strongly suggests microseconds.
        if DURATION_COL and np.nanmedian(dur) > 1e4: dur_sec = dur / 1e6
        else: dur_sec = dur
        completion = tmp['_time'] + pd.to_timedelta(np.maximum(dur_sec, 0), unit='s')
        tmp['_complete'] = completion
        # Never group across a large capture gap: each split is already chronological; large gaps create new sessions.
        gap = tmp['_complete'].diff().dt.total_seconds().fillna(0)
        session = (gap > max(3600, window_minutes*60*10)).cumsum()
        tmp['_session'] = session
        anchor = tmp.groupby('_session')['_complete'].transform('min')
        tmp['_win'] = ((tmp['_complete'] - anchor).dt.total_seconds() // (window_minutes*60)).astype(int)
    else:
        tmp['_session'] = 0
        chunk = max(min_support, int(len(tmp) / max(1, len(tmp)//min_support)))
        tmp['_win'] = (np.arange(len(tmp)) // chunk).astype(int)
    # Include session in window key so no window crosses a capture gap.
    keys = list(zip(tmp['_session'].astype(int), tmp['_win'].astype(int)))
    window_ids = pd.factorize(pd.Series(keys, dtype=object), sort=False)[0]
    tmp['_window_id'] = window_ids
    windows=[]
    for wid, idx in tmp.groupby('_window_id', sort=False).groups.items():
        ix = np.asarray(list(idx), dtype=int)
        counts = len(ix)
        windows.append({'window_id': int(wid), 'indices': ix, 'n_flows': counts,
                        'support': counts >= min_support,
                        'start_time': tmp.loc[ix, '_time'].min(),
                        'y_rate': float(y[ix].mean()),
                        'family_counts': pd.Series(families[ix]).value_counts().to_dict()})
    return tmp, windows

train_meta, train_windows = make_windows(train_df, X_train, y_train, family_train, WINDOW_MINUTES, MIN_WINDOW_SUPPORT)
val_meta, val_windows = make_windows(val_df, X_val, y_val, family_val, WINDOW_MINUTES, MIN_WINDOW_SUPPORT)
test_meta, test_windows = make_windows(test_df, X_test, y_test, family_test, WINDOW_MINUTES, MIN_WINDOW_SUPPORT)
print('Window counts train/val/test:', len(train_windows), len(val_windows), len(test_windows))
display(pd.DataFrame([{'window_id':w['window_id'],'flows':w['n_flows'],'support':w['support'],'attack_rate':w['y_rate']} for w in train_windows]).head(10))

## 6. Base IDS: residual MLP encoder and binary detection head
Class weights are calculated from training labels only. The auxiliary family head is used during training to encourage class-discriminative representations and is not needed for binary prediction.

In [ ]:
# Encode training families only for auxiliary training; unseen validation/test families map to -1.
family_le = LabelEncoder().fit(family_train)
fam_train = family_le.transform(family_train).astype(np.int64)
fam_map = {c:i for i,c in enumerate(family_le.classes_)}
fam_val = np.array([fam_map.get(c,-1) for c in family_val], dtype=np.int64)
fam_test = np.array([fam_map.get(c,-1) for c in family_test], dtype=np.int64)

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(dim, dim), nn.LayerNorm(dim))
    def forward(self, x): return torch.nn.functional.gelu(x + self.net(x))

class BaseIDS(nn.Module):
    def __init__(self, in_dim, latent_dim=64, n_families=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(in_dim, 128), nn.LayerNorm(128), nn.GELU(),
                                     nn.Dropout(.15), nn.Linear(128, latent_dim), nn.GELU(),
                                     ResidualBlock(latent_dim))
        self.binary_head = nn.Linear(latent_dim, 1)
        self.family_head = nn.Linear(latent_dim, max(2,n_families))
    def forward(self, x):
        z = self.encoder(x)
        return self.binary_head(z).squeeze(-1), z, self.family_head(z)

class_weights = compute_class_weight(class_weight='balanced', classes=np.array([0,1]), y=y_train)
pos_weight = torch.tensor([class_weights[1]/class_weights[0]], dtype=torch.float32, device=DEVICE)
model = BaseIDS(X_train.shape[1], latent_dim=64, n_families=len(family_le.classes_)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
ce_family = nn.CrossEntropyLoss()

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train), torch.from_numpy(fam_train))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
Xv_t = torch.from_numpy(X_val).to(DEVICE); Xt_t = torch.from_numpy(X_test).to(DEVICE)

history = {'train_loss':[], 'val_loss':[]}
best_state, best_val = None, float('inf')
for epoch in range(EPOCHS):
    model.train(); total=0; n_seen=0
    for xb,yb,fb in train_loader:
        xb=xb.to(DEVICE); yb=yb.float().to(DEVICE); fb=fb.to(DEVICE)
        optimizer.zero_grad()
        logits,z,flogits=model(xb)
        loss=bce(logits,yb)
        valid=fb>=0
        if valid.any(): loss = loss + 0.1*ce_family(flogits[valid],fb[valid])
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); optimizer.step()
        total += loss.item()*len(xb); n_seen += len(xb)
    model.eval()
    with torch.no_grad():
        vl,_,_=model(Xv_t)
        vloss=bce(vl, torch.from_numpy(y_val.astype(np.float32)).to(DEVICE)).item()
    history['train_loss'].append(total/max(n_seen,1)); history['val_loss'].append(vloss)
    if vloss < best_val:
        best_val=vloss; best_state=copy.deepcopy(model.state_dict())
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | train={history["train_loss"][-1]:.4f} | val={vloss:.4f}')
if best_state: model.load_state_dict(best_state)
plt.figure(figsize=(7,4)); plt.plot(history['train_loss'], label='train'); plt.plot(history['val_loss'], label='validation')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Base IDS training'); plt.legend(); plt.tight_layout(); plt.show()

## 7. Baseline test metrics
These are actual held-out test metrics. No target values are manually assigned.

In [ ]:
@torch.no_grad()
def predict_model(m, X, batch_size=8192):
    m.eval(); probs=[]; reps=[]
    for st in range(0,len(X),batch_size):
        xb=torch.from_numpy(X[st:st+batch_size]).to(DEVICE)
        logits,z,_=m(xb)
        probs.append(torch.sigmoid(logits).cpu().numpy()); reps.append(z.cpu().numpy())
    return np.concatenate(probs), np.concatenate(reps)

val_prob, val_repr = predict_model(model, X_val)
test_prob, test_repr = predict_model(model, X_test)
def detection_metrics(y, p, threshold=0.5):
    pred=(p>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    out={'Accuracy':accuracy_score(y,pred),'Precision_macro':precision_score(y,pred,average='macro',zero_division=0),
         'Recall_macro':recall_score(y,pred,average='macro',zero_division=0),'Macro_F1':f1_score(y,pred,average='macro',zero_division=0),
         'MCC':matthews_corrcoef(y,pred),'Attack_recall':tp/max(tp+fn,1),'FPR':fp/max(fp+tn,1),
         'Specificity':tn/max(tn+fp,1)}
    if len(np.unique(y))==2:
        out['AUROC']=roc_auc_score(y,p); out['AUPR']=average_precision_score(y,p)
    else: out['AUROC']=np.nan; out['AUPR']=np.nan
    return out
baseline_metrics = detection_metrics(y_test, test_prob, THRESHOLD)
display(pd.DataFrame([baseline_metrics], index=['Frozen IDS']).T.rename(columns={0:'Value'}))
print(classification_report(y_test, (test_prob>=THRESHOLD).astype(int), target_names=['Benign','Attack'], zero_division=0))

## 8. Drift-state representation
Window indicators combine feature-distribution JSD, representation discrepancy (RBF-MMD), prediction entropy/margin, alert-rate change, support, and delayed retrospective error when labels mature. Missing delayed evidence is explicitly flagged rather than treated as zero.

In [ ]:
def js_divergence(p, q, eps=1e-10):
    p=np.asarray(p,dtype=float); q=np.asarray(q,dtype=float)
    p=p/(p.sum()+eps); q=q/(q.sum()+eps); m=.5*(p+q)
    kl1=np.sum(np.where(p>0,p*np.log((p+eps)/(m+eps)),0))
    kl2=np.sum(np.where(q>0,q*np.log((q+eps)/(m+eps)),0))
    return float(.5*(kl1+kl2))

def window_feature_jsd(X_ref, X_cur, bins=10, max_features=30):
    vals=[]
    for j in range(min(X_ref.shape[1],max_features)):
        a=X_ref[:,j]; b=X_cur[:,j]
        lo,hi=np.nanpercentile(X_ref[:,j],[1,99])
        if not np.isfinite(lo+hi) or hi<=lo: continue
        edges=np.linspace(lo,hi,bins+1)
        pa=np.histogram(np.clip(a,lo,hi),edges)[0]+1
        pb=np.histogram(np.clip(b,lo,hi),edges)[0]+1
        vals.append(js_divergence(pa,pb))
    return float(np.mean(vals)) if vals else np.nan, float(np.max(vals)) if vals else np.nan

def rbf_mmd(X, Y, cap=256, seed=SEED):
    rng=np.random.default_rng(seed)
    if len(X)>cap: X=X[rng.choice(len(X),cap,replace=False)]
    if len(Y)>cap: Y=Y[rng.choice(len(Y),cap,replace=False)]
    if len(X)<2 or len(Y)<2: return np.nan
    allx=np.vstack([X,Y])
    d=pairwise_distances(allx, metric='euclidean')
    positive=d[d>0]
    sigma=np.median(positive) if len(positive) else 1.0
    gamma=1.0/(2*sigma*sigma+1e-8)
    Kxx=np.exp(-gamma*pairwise_distances(X,X,'sqeuclidean'))
    Kyy=np.exp(-gamma*pairwise_distances(Y,Y,'sqeuclidean'))
    Kxy=np.exp(-gamma*pairwise_distances(X,Y,'sqeuclidean'))
    return float((Kxx.sum()-np.trace(Kxx))/(len(X)*(len(X)-1)) +
                 (Kyy.sum()-np.trace(Kyy))/(len(Y)*(len(Y)-1)) -
                 2*Kxy.mean())

def entropy_margin(p):
    p=np.clip(np.asarray(p),1e-7,1-1e-7)
    ent=-(p*np.log(p)+(1-p)*np.log(1-p))
    margin=np.abs(p-.5)*2
    return float(ent.mean()), float(margin.mean()), float(np.quantile(margin,.1)), float((p>=THRESHOLD).mean())

def summarize_windows(windows, X, y, probs, reps, X_ref, rep_ref, y_ref, probs_ref):
    rows=[]
    ref_ent, ref_margin, ref_m10, ref_alert=entropy_margin(probs_ref)
    ref_loss=float(np.mean(-(y_ref*np.log(np.clip(probs_ref,1e-7,1))+(1-y_ref)*np.log(np.clip(1-probs_ref,1e-7,1)))))
    for w in windows:
        ix=w['indices']; xw=X[ix]; yw=y[ix]; pw=probs[ix]; rw=reps[ix]
        jsmean,jsmax=window_feature_jsd(X_ref,xw)
        mmd=rbf_mmd(rep_ref,rw)
        ent,margin,m10,alert=entropy_margin(pw)
        # Offline state construction may use labels only after their simulated release.
        loss=float(np.mean(-(yw*np.log(np.clip(pw,1e-7,1))+(1-yw)*np.log(np.clip(1-pw,1e-7,1))))) if len(np.unique(yw))>0 else np.nan
        rows.append({'window_id':w['window_id'],'n_flows':len(ix),'support':w['support'],
                     'jsd_mean':jsmean,'jsd_max':jsmax,'mmd':mmd,'entropy':ent,'margin':margin,
                     'margin_p10':m10,'alert_rate':alert,'alert_rate_change':alert-ref_alert,
                     'retrospective_loss_delta':loss-ref_loss,'retrospective_available':1,
                     'attack_rate':float(yw.mean())})
    return pd.DataFrame(rows)

# Training-only reference subset for monitoring.
rng=np.random.default_rng(SEED)
ref_n=min(10000,len(X_train)); ref_idx=rng.choice(len(X_train),ref_n,replace=False)
X_ref=X_train[ref_idx]; y_ref=y_train[ref_idx]
ref_prob, ref_repr=predict_model(model,X_ref)
train_state=summarize_windows(train_windows,X_train,y_train,
    predict_model(model,X_train)[0], predict_model(model,X_train)[1],X_ref,ref_repr,y_ref,ref_prob)
val_state=summarize_windows(val_windows,X_val,y_val,val_prob,val_repr,X_ref,ref_repr,y_ref,ref_prob)
test_state=summarize_windows(test_windows,X_test,y_test,test_prob,test_repr,X_ref,ref_repr,y_ref,ref_prob)
state_cols=['jsd_mean','jsd_max','mmd','entropy','margin','margin_p10','alert_rate_change','n_flows']
state_scaler=RobustScaler()
train_state[state_cols]=train_state[state_cols].replace([np.inf,-np.inf],np.nan).fillna(0)
val_state[state_cols]=val_state[state_cols].replace([np.inf,-np.inf],np.nan).fillna(0)
test_state[state_cols]=test_state[state_cols].replace([np.inf,-np.inf],np.nan).fillna(0)
state_scaler.fit(train_state[state_cols])
for s in (train_state,val_state,test_state):
    s.loc[:,state_cols]=state_scaler.transform(s[state_cols])
display(train_state.head())

## 9. SCDF: scenario-conditioned degradation forecasting
A lightweight gated recurrent encoder receives the recent drift-state history and scenario descriptor. Training targets are measured forward loss increases relative to the reference loss, using later training windows only. The scenario library is a controlled *descriptor* library; it does not create labels or claim causal effects. Quantile regression estimates median and upper-tail degradation. A validation residual quantile supplies a practical empirical upper correction.

In [ ]:
# Create chronology-safe future loss targets from train windows. Each origin predicts loss over later windows.
# Window-level measured losses are reconstructed from stored probabilities and true labels.
def window_losses(windows, y, probs):
    out=[]
    for w in windows:
        ix=w['indices']; yy=y[ix]; pp=np.clip(probs[ix],1e-7,1-1e-7)
        out.append(float(np.mean(-(yy*np.log(pp)+(1-yy)*np.log(1-pp)))))
    return np.asarray(out,dtype=np.float32)

train_probs, train_reps=predict_model(model,X_train)
train_loss_windows=window_losses(train_windows,y_train,train_probs)
train_ref_loss=float(np.mean(-(y_ref*np.log(np.clip(ref_prob,1e-7,1))+(1-y_ref)*np.log(np.clip(1-ref_prob,1e-7,1)))))
def history_tensor(state_df, cols, length=12):
    arr=state_df[cols].to_numpy(np.float32); out=[]
    for i in range(len(arr)):
        h=np.zeros((length,arr.shape[1]),np.float32)
        chunk=arr[max(0,i-length+1):i+1]
        h[-len(chunk):]=chunk
        out.append(h)
    return np.stack(out) if out else np.empty((0,length,len(cols)),np.float32)

H_train=history_tensor(train_state,state_cols,HISTORY_LENGTH)
H_val=history_tensor(val_state,state_cols,HISTORY_LENGTH)
H_test=history_tensor(test_state,state_cols,HISTORY_LENGTH)
# Three scenario types x three magnitudes; 0 = realised/no intervention descriptor.
SCENARIOS=[(0,0.0),(1,0.25),(1,0.5),(1,0.75),(2,0.25),(2,0.5),(2,0.75),(3,0.25),(3,0.5),(3,0.75)]
def scenario_vec(s):
    typ,mag=s
    one=np.zeros(4,np.float32); one[int(typ)]=1
    return np.concatenate([one,np.array([mag,0.2,0.2,0.0],np.float32)])
SCENARIO_DIM=len(scenario_vec(SCENARIOS[0]))
# Targets: multi-horizon observed degradation relative to reference, only from subsequent train windows.
HORIZONS=[1,3,6]
scdf_X=[]; scdf_S=[]; scdf_Y=[]
for origin in range(max(1,HISTORY_LENGTH), len(train_windows)-max(HORIZONS)):
    for scen in SCENARIOS:
        # Use actual future windows for realised scenario. Controlled scenarios perturb the descriptor,
        # while retaining observed future outcomes; scenario contrasts are thus a stress-conditioned proxy.
        target=[]
        for h in HORIZONS:
            future=train_loss_windows[origin+1:min(origin+h+1,len(train_loss_windows))]
            target.append(max(0.0,float(np.mean(future)-train_ref_loss)) if len(future) else 0.0)
        scdf_X.append(H_train[origin]); scdf_S.append(scenario_vec(scen)); scdf_Y.append(target)
scdf_X=np.asarray(scdf_X,np.float32); scdf_S=np.asarray(scdf_S,np.float32); scdf_Y=np.asarray(scdf_Y,np.float32)
print('SCDF training examples:', scdf_X.shape, scdf_S.shape, scdf_Y.shape)

In [ ]:
class SCDFNet(nn.Module):
    def __init__(self, state_dim, scenario_dim, hidden=32, horizons=3):
        super().__init__()
        self.gru=nn.GRU(state_dim,hidden,batch_first=True)
        self.scenario=nn.Sequential(nn.Linear(scenario_dim,16),nn.GELU())
        self.fuse=nn.Sequential(nn.Linear(hidden+16+1,64),nn.GELU(),nn.Dropout(.1),nn.Linear(64,horizons*2))
        self.horizons=horizons
    def forward(self,h,s,ref_loss):
        z,_=self.gru(h)
        f=torch.cat([z[:,-1],self.scenario(s),ref_loss.reshape(-1,1)],dim=1)
        q=self.fuse(f).view(-1,self.horizons,2)
        median=q[:,:,0]
        upper=median+torch.nn.functional.softplus(q[:,:,1])
        return median,upper

def pinball(pred,target,q):
    e=target-pred
    return torch.maximum(q*e,(q-1)*e).mean()

scdf=SCDFNet(len(state_cols),SCENARIO_DIM,horizons=len(HORIZONS)).to(DEVICE)
opt_scdf=torch.optim.AdamW(scdf.parameters(),lr=1e-3,weight_decay=1e-4)
if len(scdf_X)>=10:
    # chronological train/validation split over origins (not random row split)
    cut=max(1,int(len(scdf_X)*.85))
    tx=torch.tensor(scdf_X[:cut],device=DEVICE); ts=torch.tensor(scdf_S[:cut],device=DEVICE); ty=torch.tensor(scdf_Y[:cut],device=DEVICE)
    vx=torch.tensor(scdf_X[cut:],device=DEVICE); vs=torch.tensor(scdf_S[cut:],device=DEVICE); vy=torch.tensor(scdf_Y[cut:],device=DEVICE)
    for ep in range(8):
        scdf.train(); opt_scdf.zero_grad()
        med,up=scdf(tx,ts,torch.full((len(tx),),train_ref_loss,device=DEVICE))
        loss=pinball(med,ty,.5)+pinball(up,ty,.9)+0.2*torch.abs(med-ty).mean()
        loss.backward(); opt_scdf.step()
    scdf.eval()
    with torch.no_grad():
        vm,vu=scdf(vx,vs,torch.full((len(vx),),train_ref_loss,device=DEVICE))
        scdf_calibration_correction=float(torch.quantile(torch.clamp(vy-vu,min=0).reshape(-1),.9).cpu()) if len(vy) else 0.0
else:
    scdf_calibration_correction=0.0
print('SCDF empirical upper correction:', scdf_calibration_correction)

def scdf_forecast(history, scenario=(0,0.0)):
    scdf.eval()
    with torch.no_grad():
        h=torch.tensor(history[None],dtype=torch.float32,device=DEVICE)
        s=torch.tensor(scenario_vec(scenario)[None],dtype=torch.float32,device=DEVICE)
        med,up=scdf(h,s,torch.tensor([train_ref_loss],device=DEVICE))
    return med.cpu().numpy()[0], (up.cpu().numpy()[0]+scdf_calibration_correction)

## 10. SKTG: concept prototypes, dual evidence, directed interference and retention priorities
Historical attack families are represented by training-only latent prototypes. Representation tension is an unsupervised warning; behavioural tension is updated only when delayed labels are released. Edges are formed from training representation overlap and bounded propagation limits runaway graph effects. Unknown/unseen families are reported as unresolved.

In [ ]:
# Build prototype memory from training-only representations.
train_family_names=np.asarray(family_train)
prototypes={}; covariances={}; family_counts={}
for fam in np.unique(train_family_names):
    ix=np.where(train_family_names==fam)[0]
    # Cap memory to keep the prototype build tractable.
    if len(ix)>5000: ix=rng.choice(ix,5000,replace=False)
    Z=train_reps[ix] if 'train_reps' in globals() else train_repr[ix] if 'train_repr' in globals() else None
    if Z is None:
        _,all_train_reps=predict_model(model,X_train)
        train_reps=all_train_reps
        Z=train_reps[ix]
    prototypes[fam]=Z.mean(axis=0)
    try: covariances[fam]=LedoitWolf().fit(Z).covariance_
    except Exception: covariances[fam]=np.eye(Z.shape[1],dtype=np.float32)
    family_counts[fam]=len(ix)
concept_names=list(prototypes)
# Historical confusion/interference edges: nearest prototype assignment on training representations.
_, train_reps=predict_model(model,X_train)
proto_mat=np.stack([prototypes[c] for c in concept_names])
dist=pairwise_distances(train_reps,proto_mat)
nearest=np.argmin(dist,axis=1)
edge_counts=np.zeros((len(concept_names),len(concept_names)),dtype=float)
for i,fam in enumerate(train_family_names):
    src=concept_names.index(fam) if fam in concept_names else None
    dst=int(nearest[i])
    if src is not None: edge_counts[src,dst]+=1
edge=np.zeros_like(edge_counts)
for i in range(len(concept_names)):
    row=edge_counts[i]
    if row.sum()>0:
        rates=row/row.sum()
        # Only non-self edges with evidence above the historical overlap threshold propagate tension.
        for j in range(len(concept_names)):
            if i!=j and rates[j] >= 0.05: edge[i,j]=min(1.0,rates[j])
# Row-normalize propagation weights; propagation capped to two hops.
row_sums=edge.sum(axis=1,keepdims=True)
edge_norm=np.divide(edge,row_sums,out=np.zeros_like(edge),where=row_sums>0)

historical_error={}
for fam in concept_names:
    ix=np.where(train_family_names==fam)[0]
    historical_error[fam]=float(np.mean((test_prob[:0] if False else train_probs[ix]>=THRESHOLD) != y_train[ix]))

def sktg_assess(X, reps, probs, labels=None, labels_available=False):
    d=pairwise_distances(reps,proto_mat)
    nearest_idx=np.argmin(d,axis=1)
    result=[]
    for i in range(len(X)):
        fam=concept_names[int(nearest_idx[i])]
        # Soft representation tension: distance relative to the 99th-percentile training distance.
        refd=dist[:,int(nearest_idx[i])]
        scale=max(float(np.quantile(refd,.99)),1e-6)
        rep_tension=float(np.clip(d[i,nearest_idx[i]]/scale,0,3)/3)
        # Behavioural evidence is only enabled for explicitly released labels.
        beh_tension=np.nan
        if labels_available and labels is not None:
            err=int((probs[i]>=THRESHOLD)!=int(labels[i]))
            base=historical_error.get(fam,0.5)
            # Smoothed posterior-like excess-error score, not a formal Beta-CDF implementation.
            beh_tension=float(np.clip((err+base)/(1+base)-base,0,1))
        result.append({'assigned_concept':fam,'representation_tension':rep_tension,
                       'behavioural_tension':beh_tension,'evidence_verified':bool(labels_available),
                       'unresolved':bool(d[i,nearest_idx[i]]>scale)})
    out=pd.DataFrame(result)
    # Window-level graph propagation of mean concept tensions.
    own=np.zeros(len(concept_names))
    for j,c in enumerate(concept_names):
        vals=out.loc[out.assigned_concept==c,'representation_tension']
        own[j]=float(vals.mean()) if len(vals) else 0.0
    propagated=np.minimum(1.0,own + .5*(edge_norm.T@own) + .25*(edge_norm.T@edge_norm.T@own))
    rel=1-propagated
    out['concept_reliability']=[float(rel[concept_names.index(c)]) for c in out.assigned_concept]
    out['retention_priority']=[float((1-rel[concept_names.index(c)])*(1/np.sqrt(max(family_counts.get(c,1),1)))) for c in out.assigned_concept]
    return out

sktg_val=sktg_assess(X_val,val_repr,val_prob)
sktg_test=sktg_assess(X_test,test_repr,test_prob)
display(sktg_test.head())
print('Concept nodes:', len(concept_names), '| directed edges:', int((edge>0).sum()))

## 11. RBPA: risk-budgeted action selection, candidate safety gate, delayed labels and rollback
Available actions are monitoring, threshold recalibration, head-only incremental adaptation, and full retraining. The selection score combines forecast loss, estimated forgetting, compute cost, and uncertainty, subject to resource/forgetting/harm limits. Candidate updates are evaluated on a held-out safety buffer and rejected if the measured loss or false-positive rate violates the configured gate.

In [ ]:
def weighted_log_loss(y, p):
    p=np.clip(np.asarray(p),1e-7,1-1e-7); y=np.asarray(y)
    w=np.where(y==1,class_weights[1],class_weights[0])
    return float(np.average(-(y*np.log(p)+(1-y)*np.log(1-p)),weights=w))

def choose_threshold_with_fpr(y, p, fpr_cap=0.05):
    thresholds=np.unique(np.r_[0.0,p,1.0])
    best=THRESHOLD; best_recall=-1
    for t in thresholds:
        pred=(p>=t).astype(int)
        tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
        fpr=fp/max(fp+tn,1); rec=tp/max(tp+fn,1)
        if fpr<=fpr_cap and rec>best_recall: best=t; best_recall=rec
    return float(best)

def clone_model(m):
    n=BaseIDS(X_train.shape[1],64,len(family_le.classes_)).to(DEVICE)
    n.load_state_dict(copy.deepcopy(m.state_dict()))
    return n

def fit_candidate(base, X_fit, y_fit, fam_fit, mode='head', epochs=2):
    cand=clone_model(base)
    if mode=='head':
        for p in cand.encoder.parameters(): p.requires_grad=False
        params=list(cand.binary_head.parameters())
    else:
        for p in cand.parameters(): p.requires_grad=True
        params=list(cand.parameters())
    opt=torch.optim.AdamW(params,lr=5e-4,weight_decay=1e-4)
    ds=TensorDataset(torch.tensor(X_fit,dtype=torch.float32),torch.tensor(y_fit,dtype=torch.float32),
                     torch.tensor(np.maximum(fam_fit,0),dtype=torch.long))
    loader=DataLoader(ds,batch_size=min(BATCH_SIZE,max(16,len(ds))),shuffle=True)
    for _ in range(epochs):
        cand.train()
        for xb,yb,fb in loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE)
            opt.zero_grad(); logits,z,fl=cand(xb); loss=bce(logits,yb)
            loss.backward(); opt.step()
    cand.eval()
    return cand

# A demonstrator decision on the validation segment; in online deployment only matured labels may be used.
val_base_loss=weighted_log_loss(y_val,val_prob)
val_tension=float(sktg_val['retention_priority'].mean()) if len(sktg_val) else 0.0
forecast_med, forecast_up=scdf_forecast(H_val[-1] if len(H_val) else np.zeros((HISTORY_LENGTH,len(state_cols))))
scenarios_forecasts=[scdf_forecast(H_val[-1],s) for s in SCENARIOS] if len(H_val) else [(np.zeros(len(HORIZONS)),np.zeros(len(HORIZONS)))]
forecast_risk=max(float(np.max(u)) for _,u in scenarios_forecasts)
print('Validation weighted loss:', val_base_loss, '| SCDF worst upper risk:', forecast_risk, '| SKTG mean priority:', val_tension)

# Chronological safety split inside validation; if tiny, use the final 20% of training as safety.
safe_start=int(len(X_val)*0.5)
X_adapt, y_adapt, f_adapt = X_val[:safe_start], y_val[:safe_start], fam_val[:safe_start]
X_safe, y_safe, f_safe = X_val[safe_start:], y_val[safe_start:], fam_val[safe_start:]
if len(X_safe)<20 or len(np.unique(y_safe))<2:
    split=int(len(X_train)*0.8)
    X_adapt,y_adapt,f_adapt=X_train[:split],y_train[:split],fam_train[:split]
    X_safe,y_safe,f_safe=X_train[split:],y_train[split:],fam_train[split:]
print('Adapt fitting rows:',len(X_adapt),'Safety rows:',len(X_safe),'Safety labels:',np.unique(y_safe))

In [ ]:
def safe_gate(y, base_p, cand_p, fpr_cap=0.08, max_loss_increase=HARM_TOLERANCE):
    base_loss=weighted_log_loss(y,base_p); cand_loss=weighted_log_loss(y,cand_p)
    base_pred=base_p>=THRESHOLD; cand_pred=cand_p>=THRESHOLD
    tn,fp,fn,tp=confusion_matrix(y,cand_pred,labels=[0,1]).ravel()
    fpr=fp/max(fp+tn,1)
    passed=(cand_loss-base_loss <= max_loss_increase) and (fpr<=fpr_cap)
    return {'accepted':bool(passed),'base_loss':base_loss,'candidate_loss':cand_loss,
            'loss_delta':cand_loss-base_loss,'candidate_fpr':fpr,
            'reason':'accepted' if passed else ('loss harm exceeded tolerance' if cand_loss-base_loss>max_loss_increase else 'FPR cap exceeded')}

# Score feasible actions. The cost/forgetting terms are explicit approximations calibrated as engineering proxies.
action_candidates=[]
base_safe_prob,_=predict_model(model,X_safe)
base_safe_loss=weighted_log_loss(y_safe,base_safe_prob)
action_candidates.append({'action':'monitor','candidate':model,'threshold':THRESHOLD,
                          'score':val_base_loss+0.15*val_tension+0.05*forecast_risk,
                          'resource_cost':0.05,'forgetting_est':0.0,'harm_est':0.0,
                          'gate':{'accepted':True,'reason':'monitoring baseline'}})

# Threshold recalibration on adaptation buffer only.
if len(X_adapt) and len(np.unique(y_adapt))==2:
    p_adapt,_=predict_model(model,X_adapt)
    new_thr=choose_threshold_with_fpr(y_adapt,p_adapt,fpr_cap=0.05)
    p_safe,_=predict_model(model,X_safe)
    old_thr=THRESHOLD
    pred_loss=weighted_log_loss(y_safe,p_safe)
    action_candidates.append({'action':'recalibrate','candidate':model,'threshold':new_thr,
                              'score':pred_loss+0.10*val_tension+0.03,
                              'resource_cost':0.10,'forgetting_est':0.01,'harm_est':abs(new_thr-old_thr)*0.05,
                              'gate':{'accepted':True,'reason':'threshold-only candidate'}})

# Adaptation candidates require both labels in the fitting buffer.
if len(X_adapt)>=2*MIN_CLASS_SUPPORT_FOR_UPDATE and len(np.unique(y_adapt))==2 and len(X_safe)>0 and len(np.unique(y_safe))==2:
    famfit=np.where(f_adapt>=0,f_adapt,0)
    for mode,action,cost,forget in [('head','incremental_head_update',0.35,0.08),
                                    ('full','full_retraining',0.65,0.18)]:
        try:
            cand=fit_candidate(model,X_adapt,y_adapt,famfit,mode=mode,epochs=2 if mode=='head' else 3)
            cand_prob,_=predict_model(cand,X_safe)
            gate=safe_gate(y_safe,base_safe_prob,cand_prob)
            # The objective is a conservative proxy; gate still independently decides deployment.
            cand_loss=weighted_log_loss(y_safe,cand_prob)
            score=cand_loss + .15*min(forget,1) + .10*cost + .10*forecast_risk + .05*val_tension
            harm=max(0.0,cand_loss-base_safe_loss)
            feasible=(cost<=RESOURCE_BUDGET and forget<=FORGETTING_BUDGET and harm<=HARM_TOLERANCE)
            action_candidates.append({'action':action,'candidate':cand,'threshold':THRESHOLD,
                                      'score':score,'resource_cost':cost,'forgetting_est':forget,
                                      'harm_est':harm,'gate':gate,'feasible':feasible})
        except Exception as e:
            print('Candidate skipped:',action,str(e)[:160])

for a in action_candidates:
    a.setdefault('feasible',True)
eligible=[a for a in action_candidates if a['feasible'] and a['resource_cost']<=RESOURCE_BUDGET
          and a['forgetting_est']<=FORGETTING_BUDGET and a['harm_est']<=HARM_TOLERANCE]
selected=min(eligible,key=lambda a:a['score']) if eligible else action_candidates[0]
deployed_model=model; deployed_threshold=selected['threshold']; previous_model=model
if selected['action'] not in ('monitor','recalibrate') and selected['gate'].get('accepted',False):
    deployed_model=selected['candidate']
elif selected['action'] not in ('monitor','recalibrate') and not selected['gate'].get('accepted',False):
    selected['deployment_result']='rejected_by_safety_gate'
print('Selected action:',selected['action'])
print('Deployment gate:',selected['gate'])
print('Candidate table:')
display(pd.DataFrame([{k:v for k,v in a.items() if k not in ('candidate','gate')} | {'gate':a['gate'].get('reason','')} for a in action_candidates]))

## 12. Delayed-feedback chronological simulation and probation rollback
Predictions are stored before labels are released. Labels become available after `LABEL_DELAY_WINDOWS`; only then are retrospective errors recorded. Updates remain pending during probation and can be rolled back if delayed probation loss violates the configured margin. This is a compact simulation; it does not tune the policy on test labels.

In [ ]:
# Sequential test stream: predictions are generated by the currently deployed revision.
# Labels are accessed only after the configured simulated delay for retrospective audit.
online_rows=[]
pending=[]
current_model=deployed_model
current_threshold=deployed_threshold
probation=None
rollback_events=[]
for wi,w in enumerate(test_windows):
    ix=w['indices']
    p,_=predict_model(current_model,X_test[ix])
    rec={'window':wi,'window_id':w['window_id'],'n_flows':len(ix),'prob':p.copy(),
         'model_revision':0 if current_model is deployed_model else 1,'threshold':current_threshold}
    pending.append((wi,rec, y_test[ix].copy()))
    # Release labels for records whose delay has matured.
    released=[item for item in pending if wi-item[0]>=LABEL_DELAY_WINDOWS]
    pending=[item for item in pending if wi-item[0]<LABEL_DELAY_WINDOWS]
    for origin,rr,yy in released:
        pred=(rr['prob']>=rr['threshold']).astype(int)
        rr['released_at_window']=wi
        rr['retrospective_accuracy']=float(accuracy_score(yy,pred)) if len(yy) else np.nan
        rr['retrospective_loss']=weighted_log_loss(yy,rr['prob']) if len(yy) else np.nan
        online_rows.append({k:v for k,v in rr.items() if k!='prob'})
    # A probation record is audited only after its delayed labels mature.
    if probation is not None and wi>=probation['audit_at']:
        audit_p,audit_y=probation['prob'],probation['y']
        audit_loss=weighted_log_loss(audit_y,audit_p) if len(audit_y) else np.nan
        if np.isfinite(audit_loss) and audit_loss > probation['baseline_loss'] + 0.02:
            current_model=probation['previous_model']; current_threshold=probation['previous_threshold']
            rollback_events.append({'window':wi,'reason':'probation loss exceeded rollback margin','audit_loss':audit_loss})
        probation=None

# Release remaining labels at end of stream for reporting only (not for test-time policy selection).
for origin,rr,yy in pending:
    rr['released_at_window']=len(test_windows)-1+LABEL_DELAY_WINDOWS
    rr['retrospective_accuracy']=float(accuracy_score(yy,(rr['prob']>=rr['threshold']).astype(int))) if len(yy) else np.nan
    rr['retrospective_loss']=weighted_log_loss(yy,rr['prob']) if len(yy) else np.nan
    online_rows.append({k:v for k,v in rr.items() if k!='prob'})
online_log=pd.DataFrame(online_rows)
print('Delayed-feedback log rows:',len(online_log))
print('Rollback events:',rollback_events)
display(online_log.head())

## 13. Final test evaluation and visualisations
Report detection, drift, knowledge tension and adaptation logs separately. The model's metrics are computed on the held-out test labels; policy thresholds and model configuration are not retuned on those outcomes.

In [ ]:
final_prob, final_repr=predict_model(deployed_model,X_test)
final_metrics=detection_metrics(y_test,final_prob,deployed_threshold)
display(pd.DataFrame([baseline_metrics,final_metrics],index=['Initial frozen IDS','Final deployed model']).T)
fig,ax=plt.subplots(1,2,figsize=(12,4))
fpr,tpr,_=roc_curve(y_test,final_prob)
ax[0].plot(fpr,tpr,label=f'AUROC={final_metrics["AUROC"]:.3f}'); ax[0].plot([0,1],[0,1],'--')
ax[0].set_xlabel('False positive rate'); ax[0].set_ylabel('True positive rate'); ax[0].set_title('Test ROC'); ax[0].legend()
prec,rec,_=precision_recall_curve(y_test,final_prob)
ax[1].plot(rec,prec,label=f'AUPR={final_metrics["AUPR"]:.3f}')
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision'); ax[1].set_title('Test precision-recall'); ax[1].legend()
plt.tight_layout(); plt.show()

fig,ax=plt.subplots(1,2,figsize=(13,4))
if len(train_state):
    ax[0].plot(train_state['jsd_mean'].to_numpy(),label='JSD mean')
    ax[0].plot(train_state['entropy'].to_numpy(),label='Prediction entropy')
    ax[0].set_title('Training drift-state indicators'); ax[0].set_xlabel('Window'); ax[0].legend()
if len(sktg_test):
    sktg_test['representation_tension'].rolling(max(1,min(20,len(sktg_test))),min_periods=1).mean().plot(ax=ax[1])
    ax[1].set_title('SKTG representation tension'); ax[1].set_xlabel('Test flow index')
plt.tight_layout(); plt.show()

cm=confusion_matrix(y_test,(final_prob>=deployed_threshold).astype(int),labels=[0,1])
fig,ax=plt.subplots(figsize=(5,4)); im=ax.imshow(cm,interpolation='nearest')
ax.set_title('Test confusion matrix'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_xticks([0,1],['Benign','Attack']); ax.set_yticks([0,1],['Benign','Attack'])
for i in range(2):
    for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center')
plt.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

## 14. Cost, drift and adaptation summary
Latency is measured empirically on a sample and model size is computed from parameter count. Values are environment-specific and should be reported with hardware details. Unsupported forecasts, low-support windows, false alarms, and rollback events should be included in the final paper tables.

In [ ]:
# Empirical inference latency and parameter count.
sample_n=min(4096,len(X_test))
if sample_n:
    xb=torch.from_numpy(X_test[:sample_n]).to(DEVICE)
    for _ in range(3):
        with torch.no_grad(): deployed_model(xb)
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    t0=time.perf_counter()
    reps=20
    with torch.no_grad():
        for _ in range(reps): deployed_model(xb)
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    latency_ms=(time.perf_counter()-t0)*1000/(reps*sample_n)
else: latency_ms=np.nan
n_params=sum(p.numel() for p in deployed_model.parameters())
model_size_mb=sum(p.numel()*p.element_size() for p in deployed_model.parameters())/(1024**2)
summary={
    'test_flows':int(len(y_test)),
    'test_windows':int(len(test_windows)),
    'low_support_test_windows':int(sum(not w['support'] for w in test_windows)),
    'selected_action':selected['action'],
    'selected_action_gate':selected['gate'].get('reason',''),
    'rollback_count':len(rollback_events),
    'inference_latency_ms_per_flow':float(latency_ms),
    'parameter_count':int(n_params),
    'parameter_size_MB':float(model_size_mb),
    'privacy_budget':None, # no differential-privacy mechanism is implemented in this reference notebook
    'communication_volume_bytes':0, # single-node simulation; no federated clients are instantiated
}
display(pd.Series(summary,name='Value').to_frame())
print('Interpretation: privacy budget is N/A because this is not a differentially-private or federated implementation.')

## 15. Export result tables (optional)
This saves only CSV result tables, not model checkpoints. In Colab, files are written to the current runtime. No model is saved.

In [ ]:
# Uncomment to export tables:
# pd.DataFrame([baseline_metrics, final_metrics], index=['Frozen IDS','Final deployed IDS']).to_csv('detection_metrics.csv')
# train_state.to_csv('train_drift_state.csv', index=False)
# val_state.to_csv('validation_drift_state.csv', index=False)
# test_state.to_csv('test_drift_state.csv', index=False)
# sktg_test.to_csv('sktg_test_tensions.csv', index=False)
# online_log.to_csv('delayed_feedback_log.csv', index=False)
# print('CSV exports written to current working directory.')

## Method-to-code map and reproducibility cautions

- **Chronological preparation:** Sections 3.2–3.3; train-only imputation/scaling/encoding; window support flags.
- **Base IDS:** residual MLP encoder, binary head, training-only auxiliary family head.
- **SCDF reference:** gated recurrent history encoder, scenario descriptors, multi-horizon median/upper quantiles, validation residual correction.
- **SKTG reference:** prototype memory, representation tension, delayed behavioural evidence hook, directed overlap graph, bounded propagation, retention priority.
- **RBPA reference:** monitoring/recalibration/head update/full retraining candidates, cost/forgetting/harm budgets, independent safety gate.
- **Operational protocol:** delayed label queue, prediction revision log, probation/rollback scaffold.

For a publication-grade replication, use the exact capture-session boundaries specified in the manuscript, run forward-chained out-of-fold models and rolling-origin shadow updates, fit SKTG isotonic maps and RBPA quantile surrogates on their designated validation segments, apply moving-block bootstrap calibration, and repeat the locked experiment over the specified independent seeds. Do not present proxy components in this notebook as exact replications of those procedures.